In [0]:
dbutils.widgets.text("catalog", "banking")
dbutils.widgets.text("schema_landing", "landing")
dbutils.widgets.text("schema_bronze", "bronze")
dbutils.widgets.text("schema_silver", "silver")
dbutils.widgets.text("schema_gold", "gold")

catalog = dbutils.widgets.get("catalog")
schema_landing = dbutils.widgets.get("schema_landing")
schema_bronze = dbutils.widgets.get("schema_bronze")
schema_silver = dbutils.widgets.get("schema_silver")
schema_gold = dbutils.widgets.get("schema_gold")

silver_customers_table = f"{catalog}.{schema_silver}.silver_customers"
silver_transactions_table = f"{catalog}.{schema_silver}.silver_transactions"
silver_accounts_table = f"{catalog}.{schema_silver}.silver_accounts"
silver_credit_table = f"{catalog}.{schema_silver}.silver_credit"
gold_table_full = f"{catalog}.{schema_gold}.gold_daily_bank_kpi"

In [0]:
import pyspark.sql.functions as F

txn_daily = (
    spark.table(silver_transactions_table)
    .groupBy(F.to_date("txn_timestamp").alias("txn_date"))
    .agg(
        F.count("txn_id").alias("total_transactions"),
        F.sum("amount").alias("total_transaction_amount")
    )
)

customer_metrics = (
    spark.table(silver_customers_table)
    .agg(
        F.countDistinct("customer_id").alias("total_customers")
    )
)

account_metrics = (
    spark.table(silver_accounts_table)
    .agg(
        F.count("account_id").alias("total_accounts"),
        F.sum("balance").alias("total_balance")
    )
)

credit_metrics = (
    spark.table(silver_credit_table)
    .agg(
        F.avg("credit_score").alias("avg_credit_score"),
        F.sum(
            F.when(F.col("risk_grade") == "HIGH", 1).otherwise(0)
        ).alias("high_risk_customers")
    )
)

# Cross join all metrics to txn_daily
result = txn_daily.crossJoin(customer_metrics).crossJoin(account_metrics).crossJoin(credit_metrics)

result.write.format("delta").mode("overwrite").saveAsTable(gold_table_full)

In [0]:
count = spark.sql("""
SELECT COUNT(*) AS cnt
FROM banking.gold.gold_daily_bank_kpi
""").collect()[0]["cnt"]

dbutils.notebook.exit(str(count))